# ШАГ 1: Генерация поведенческого датасета и поиск Якорей среды

In [ ]:
import pandas as pd
import numpy as np

# 1. ГЕНЕРАЦИЯ СИНТЕТИЧЕСКОГО ДАТАСЕТА "АВТОСАЛОН ЭЛЕКТРОМОБИЛЕЙ"
np.random.seed(42)
n_samples = 1000

data = {
    'Доход_тыс_руб': np.random.normal(150, 80, n_samples),  # Непрерывный признак
    'Возраст': np.random.randint(16, 75, n_samples),         # Непрерывный признак
    'Инфраструктура_EV': np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7]) # Есть ли зарядки рядом
}

df = pd.DataFrame(data)

# Логика скрытых триггеров (целевая переменная Купил/Не купил)
y = []
for idx, row in df.iterrows():
    # Анти-Якорь (Фильтр отсечки): несовершеннолетние или нулевой доход
    if row['Возраст'] < 18 or row['Доход_тыс_руб'] <= 30:
        prob = 0.01  # Шанс покупки стремится к нулю, даже не зайдут в салон
    # Положительный Якорь: высокий доход + есть зарядки
    elif row['Доход_тыс_руб'] > 300 and row['Инфраструктура_EV'] == 1:
        prob = 0.95  # Гарантированная покупка
    # Зона Блефа (Сомневающиеся): средний доход, решение зависит от других факторов
    else:
        prob = 0.48  # Максимальная энтропия, классические 50/50
        
    y.append(np.random.choice([0, 1], p=[1 - prob, prob]))

df['Купил_электромобиль'] = y
print(f"[+] Базовый датасет успешно сгенерирован! Размерность: {df.shape}")

In [ ]:
# 2. АЛГОРИТМ ИДЕНТИФИКАЦИИ ЯКОРЕЙ И АНТИ-ЯКОРЕЙ
def identify_anchors_and_filters(df, continuous_cols, target_col, bluff_threshold=0.15):
    positive_anchors = {}
    anti_anchors = {}
    bluff_zones = {}
    
    print("=== АУДИТ СРЕДНЕГО ПОЛЯ: СЕПАРАЦИЯ ПРИЗНАКОВ ===\n")
    
    for col in continuous_cols:
        df_temp = df.copy()
        # Бьем на 5 квантилей
        df_temp['bin'] = pd.qcut(df_temp[col], q=5, duplicates='drop', labels=False)
        stats = df_temp.groupby('bin')[target_col].agg(['mean', 'count'])
        
        for bin_idx, row in stats.iterrows():
            prob = row['mean']
            bin_data = df_temp[df_temp['bin'] == bin_idx][col]
            min_val, max_val = bin_data.min(), bin_data.max()
            interval_str = f"[{min_val:.1f} - {max_val:.1f}] (P={prob:.2f})"
            
            if prob >= 0.80:
                if col not in positive_anchors: positive_anchors[col] = []
                positive_anchors[col].append(interval_str)
            elif prob <= 0.10:
                if col not in anti_anchors: anti_anchors[col] = []
                anti_anchors[col].append(interval_str)
            elif 0.5 - bluff_threshold < prob < 0.5 + bluff_threshold:
                if col not in bluff_zones: bluff_zones[col] = []
                bluff_zones[col].append(interval_str)
                
    print("[+] ПОЛОЖИТЕЛЬНЫЕ ЯКОРЯ:")
    for k, v in positive_anchors.items(): print(f"  • {k}: {v}")
    print("\n[-] АНТИ-ЯКОРЯ:")
    for k, v in anti_anchors.items(): print(f"  • {k}: {v}")
    print("\n[?] ЗОНЫ БЛЕФА (МАКС. ЭНТРОПИЯ):")
    for k, v in bluff_zones.items(): print(f"  • {k}: {v}")
    
    return positive_anchors, anti_anchors, bluff_zones

# Запуск анализа
pos, anti, bluff = identify_anchors_and_filters(df, ['Доход_тыс_руб', 'Возраст'], 'Купил_электромобиль')